In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
isin = pd.read_excel('Data\\EA_ISINs.xlsx')

In [3]:
unique_isin = tuple(isin['ISIN'])

In [4]:
isin['ISIN'].str[:2].unique()

array(['DE', 'IT', 'FR', 'ES'], dtype=object)

In [5]:
treasury = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [6]:
unique_treasury = tuple(treasury['ISIN'].unique())

In [7]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [8]:
hf_overlap = tuple(hedge_funds['entity_id'].unique())

In [ ]:
# Data prep
query = f"""

SELECT 
    s.lender_id AS dealer_id,
    s.lender_name AS dealer_name, 
    COUNT(*) as cnt
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.lender_id, s.lender_name
ORDER BY s.lender_id, s.lender_name

"""

df_lending_d = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_25356\464336709.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending_d = pd.read_sql_query(query, cnxn)


In [ ]:
# Data prep
query = f"""

SELECT 
    s.borrower_id AS dealer_id,
    s.borrower_name AS dealer_name, 
    COUNT(*) as cnt
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.borrower_id, s.borrower_name
ORDER BY s.borrower_id, s.borrower_name
"""

df_borrowing_d = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_25356\2310026352.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing_d = pd.read_sql_query(query, cnxn)


In [27]:
df_d = pd.concat([df_lending_d, df_borrowing_d])[['dealer_id', 'dealer_name']].drop_duplicates().reset_index(drop=True)

In [28]:
df_borrowing_d['dealer_id'].unique()

array(['54930056FHWP7GIWYY08', '5493006QMFDDMYWIAM13',
       '549300FH0WJAPEHTIQ77', '549300ZK53CNGEEI6A29',
       '7LTWFZYICNSX8D621K86', 'DGQCSV2PHVF7I2743539',
       'K6Q0W1PS1L1O4IQL9C32', 'KX1WK48MPD4Y2NCUIZ63',
       'O2RNE8IBXP4R0TD8PU41', 'R0MUWSFPU8MPRO8K5P83',
       'RRAN7P32P0W0YY4XQW79', 'XKZZ2JZF41MRHTR1V493'], dtype=object)

In [29]:
df_d.to_excel('dealer_list.xlsx')

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS borrowing_volume, 
    AVG(haircut)                AS borrowing_haircut, 
    AVG(CASE WHEN contractual_maturity <= 0 THEN 1 ELSE contractual_maturity END) AS borrowing_tenor
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id
ORDER BY s.business_date, s.borrower_id, s.lender_id

"""

df_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19648\3176663820.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing = pd.read_sql_query(query, cnxn)


In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS lending_volume, 
    AVG(haircut)                AS lending_haircut, 
    AVG(CASE WHEN contractual_maturity <= 0 THEN 1 ELSE contractual_maturity END) AS lending_tenor
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id
ORDER BY s.business_date, s.lender_id, s.borrower_id
"""

df_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19648\1164294200.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending = pd.read_sql_query(query, cnxn)


In [31]:
df = df_borrowing.merge(df_lending, on= ['business_date', 'fund_id', 'dealer_id'], how = 'outer')

In [32]:
df.to_csv('key dataframe\\fund_dealer_day.csv')

In [41]:
cds_map = pd.read_excel('Data\\dealer_bloomberg.xlsx', sheet_name='Sheet1')
cds_wide = pd.read_excel('Data\\dealer_bloomberg.xlsx', sheet_name='Sheet3')

In [42]:
cds_wide = cds_wide.loc[:, ~cds_wide.columns.str.endswith('.1')] # tickers shared by two LEIs were pulled twice
cds_long = cds_wide.melt(id_vars='Dates', var_name='Bloomberg', value_name='cds')
cds_long = cds_long.merge(cds_map[['dealer_id', 'Bloomberg', 'nationality']], on='Bloomberg', how='inner') # every LEI gets its parent's series
cds_long['period'] = cds_long['Dates'].dt.strftime('%Y-%m-%d')
cds_long = cds_long[['dealer_id', 'Bloomberg', 'nationality', 'period', 'cds']].rename(columns={'Bloomberg': 'bloomberg'})

In [43]:
cds_long

,dealer_id,bloomberg,nationality,period,cds
0,1VUV7VQFKUOQSJ21A208,ACAFP A CDS EUR SR 5Y D14 Corp,FR,2021-01-01,52.710
1,96950023SCR9X9F3L662,ACAFP A CDS EUR SR 5Y D14 Corp,FR,2021-01-01,52.710
2,1VUV7VQFKUOQSJ21A208,ACAFP A CDS EUR SR 5Y D14 Corp,FR,2021-01-04,52.385
3,96950023SCR9X9F3L662,ACAFP A CDS EUR SR 5Y D14 Corp,FR,2021-01-04,52.385
4,1VUV7VQFKUOQSJ21A208,ACAFP A CDS EUR SR 5Y D14 Corp,FR,2021-01-05,53.189
...,...,...,...,...,...
48835,XKZZ2JZF41MRHTR1V493,CINC CDS USD SR 5Y D14 Corp,US,2026-08-28,51.410
48836,XKZZ2JZF41MRHTR1V493,CINC CDS USD SR 5Y D14 Corp,US,2026-08-31,51.727
48837,XKZZ2JZF41MRHTR1V493,CINC CDS USD SR 5Y D14 Corp,US,2026-09-01,51.626
48838,XKZZ2JZF41MRHTR1V493,CINC CDS USD SR 5Y D14 Corp,US,2026-09-02,50.628


In [44]:
cds_long.to_csv('key dataframe\\dealer_cds.csv')

In [9]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND (s_lender.sector <> 'HF' OR s_lender.sector IS NULL)
GROUP BY s.business_date, s.borrower_id
ORDER BY s.business_date, s.borrower_id
"""

df_book_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19648\2876565924.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_book_borrowing = pd.read_sql_query(query, cnxn)


In [10]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS dealer_id,
    SUM(s.nominal_value)                                                                AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND (s_borrower.sector <> 'HF' OR s_borrower.sector IS NULL)
GROUP BY s.business_date, s.lender_id
ORDER BY s.business_date, s.lender_id
"""

df_book_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19648\1011889665.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_book_lending = pd.read_sql_query(query, cnxn)


In [11]:
df_book = df_book_borrowing.merge(df_book_lending, on= ['business_date', 'dealer_id'], how = 'outer')

In [12]:
df_book.to_csv('key dataframe\\dealer_book_day.csv')

In [ ]:
# Data prep
# rates by fund x dealer x bond x day, simple averages over the trades that carry a rate,
# the trade count lets Stata rebuild the rate sum and leave the fund out of the bond day benchmark
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    s.security_isin,
    SUM(s.nominal_value)                                                                AS borrowing_volume, 
    AVG(s.repo_rate)                                                                    AS borrowing_rate, 
    COUNT(s.repo_rate)                                                                  AS borrowing_trades
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id, s.security_isin
ORDER BY s.business_date, s.borrower_id, s.lender_id, s.security_isin

"""

df_rate_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    s.security_isin,
    SUM(s.nominal_value)                                                                AS lending_volume, 
    AVG(s.repo_rate)                                                                    AS lending_rate, 
    COUNT(s.repo_rate)                                                                  AS lending_trades
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id, s.security_isin
ORDER BY s.business_date, s.lender_id, s.borrower_id, s.security_isin
"""

df_rate_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_rate = df_rate_borrowing.merge(df_rate_lending, on= ['business_date', 'fund_id', 'dealer_id', 'security_isin'], how = 'outer')

In [ ]:
df_rate.to_csv('key dataframe\\fund_dealer_bond_day.csv')

In [ ]:
# Data prep
# the bond day benchmark over everything, all sectors, same filters otherwise,
# the counts give the cell size for a minimum number of counterparties
query = f"""

SELECT 
    s.business_date,
    s.security_isin,
    AVG(s.repo_rate)                                                                    AS market_rate, 
    COUNT(s.repo_rate)                                                                  AS market_trades, 
    COUNT(DISTINCT s.borrower_id)                                                       AS market_borrowers, 
    COUNT(DISTINCT s.lender_id)                                                         AS market_lenders
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.security_isin
ORDER BY s.business_date, s.security_isin
"""

df_market = pd.read_sql_query(query, cnxn)

In [ ]:
df_market.to_csv('key dataframe\\bond_day_rate.csv')

## Dealer fragility by collateral country

The same two queries as for the fund x dealer x day panel, with the collateral country added as the first two letters of the ISIN. The dealer book is unchanged, the treatment stays at the dealer level.

In [ ]:
# Data prep
# fund x dealer x collateral country x day, the country is the first two letters of the ISIN
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    LEFT(s.security_isin, 2) AS country,
    SUM(s.nominal_value)                                                                AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER' OR s_lender.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id, LEFT(s.security_isin, 2)
ORDER BY s.business_date, s.borrower_id, s.lender_id, LEFT(s.security_isin, 2)

"""

df_country_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    LEFT(s.security_isin, 2) AS country,
    SUM(s.nominal_value)                                                                AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER' OR s_borrower.sector = 'MFI')
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id, LEFT(s.security_isin, 2)
ORDER BY s.business_date, s.lender_id, s.borrower_id, LEFT(s.security_isin, 2)

"""

df_country_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_country = df_country_borrowing.merge(df_country_lending, on= ['business_date', 'fund_id', 'dealer_id', 'country'], how = 'outer')

In [ ]:
df_country.to_csv('key dataframe\\fund_dealer_country_day.csv')